In [ ]:
## === Import principali ===
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import glob
import gzip
import shutil
import tempfile
import h5py

# ============================================================
# LETTURA DEI FILE DELLA DAQ
#
# I file hanno due formati: v1 un dataset per evento (event0, event1, ...),
# v2 un unico dataset /events/waveforms -- nato per poter leggere mentre la
# run e' in corso, che con un dataset per evento non si puo' fare. E i canali
# registrati NON sono sempre 32: si prendono dal file, non si assumono.
# ============================================================
import contextlib, gzip, shutil, tempfile


@contextlib.contextmanager
def apri_run(path):
    """Apre un file della DAQ, compresso o no. A run finita diventa .h5.gz."""
    tmp = None
    if path.endswith(".gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False).name
        with gzip.open(path, "rb") as fin, open(tmp, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        path = tmp
    f = h5py.File(path, "r")
    try:
        yield f
    finally:
        f.close()
        if tmp:
            os.unlink(tmp)


def eventi_di(f):
    """(chiavi degli eventi, funzione che ne legge uno) per i due formati."""
    g = f["/events"]
    if "waveforms" in g:
        ds = g["waveforms"]
        return list(range(ds.shape[0])), lambda k: np.asarray(ds[k], dtype=np.float64)
    chiavi = sorted(g.keys(), key=lambda x: int(x.replace("event", "")))
    return chiavi, lambda k: np.asarray(g[k][:], dtype=np.float64)


def canali_di(f, voluti):
    """(canali nel file, indici di quelli chiesti).

    I canali si chiedono per NUMERO -- 16 e' il canale 16 del V1742 -- e non
    per posizione nella matrice: la posizione dipende da quali canali sono
    stati registrati in quella run, e cambia da una run all'altra.
    """
    lista = [int(c) for c in f["/config"].attrs["ChannelList"]]
    manca = [c for c in voluti if c not in lista]
    if manca:
        raise ValueError("Canali non nel file: %s. Il file contiene %s." % (manca, lista))
    return lista, [lista.index(c) for c in voluti]

import pandas as pd
from scipy.signal import iirnotch, filtfilt, find_peaks, butter, savgol_filter
from scipy.optimize import curve_fit
from scipy.stats import moyal


# ============================================================
# RUMORE PERIODICO — FUNZIONI
# ============================================================

def detect_noise_frequencies(wf, fs, n_peaks=6, fmin=5e6, fmax=500e6):
    W = np.fft.rfft(wf)
    freq = np.fft.rfftfreq(len(wf), d=1/fs)
    spec = np.abs(W)

    mask = (freq > fmin) & (freq < fmax)
    freq = freq[mask]
    spec = spec[mask]

    peaks, props = find_peaks(spec, height=np.max(spec) * 0.2, distance=30)

    if len(peaks) == 0:
        return []

    peak_freqs = freq[peaks]
    order = np.argsort(props["peak_heights"])[::-1]
    peak_freqs = peak_freqs[order]

    return peak_freqs[:n_peaks]


def bandstop(wf, fs, f1, f2, order=3):
    b, a = butter(order, [f1/(fs/2), f2/(fs/2)], btype='bandstop')
    return filtfilt(b, a, wf)


def preprocess_waveform(wf_mv, fs, auto_detect=True, max_notch=6, q=60,
                        rf_bands=None, sg_window=9, sg_order=2):
    if rf_bands is None:
        rf_bands = []

    wf_filt = wf_mv.copy()

    if auto_detect:
        freqs = detect_noise_frequencies(wf_mv, fs, n_peaks=max_notch)
    else:
        freqs = []

    for f0 in freqs:
        if f0 < fs / 2:
            b, a = iirnotch(f0, q, fs)
            wf_filt = filtfilt(b, a, wf_filt)

    for f1, f2 in rf_bands:
        if f2 < fs / 2:
            wf_filt = bandstop(wf_filt, fs, f1, f2)

    if sg_window >= 3 and sg_window % 2 == 1 and sg_window < len(wf_filt):
        wf_filt = savgol_filter(wf_filt, window_length=sg_window, polyorder=sg_order)

    return wf_filt


def compute_baseline_and_integral(time_ns, wf_mv, tmin_ns, tmax_ns,
                                  baseline_window_ns=None,
                                  negative_threshold_mv=-5.0):
    signal_mask = (time_ns >= tmin_ns) & (time_ns <= tmax_ns)
    if not np.any(signal_mask):
        raise ValueError(f"Finestra di integrazione vuota: [{tmin_ns}, {tmax_ns}] ns")

    if baseline_window_ns is None:
        baseline_mask = time_ns < tmin_ns
    else:
        b0, b1 = baseline_window_ns
        baseline_mask = (time_ns >= b0) & (time_ns <= b1)

    baseline = np.mean(wf_mv[baseline_mask]) if np.any(baseline_mask) else 0.0
    wf_corr = wf_mv - baseline

    wf_in_window = wf_corr[signal_mask]
    min_in_window = np.min(wf_in_window)
    passed_threshold = min_in_window <= negative_threshold_mv

    if passed_threshold:
        integral_mv_ns = -np.trapezoid(wf_in_window, time_ns[signal_mask])
    else:
        integral_mv_ns = 0.0

    return baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window


# ============================================================
# FIT TIPO LANDAU (approssimazione con Moyal)
# ============================================================

def landau_like_pdf(x, amplitude, mpv, sigma):
    sigma = np.abs(sigma)
    return amplitude * moyal.pdf(x, loc=mpv, scale=sigma)


def fit_landau_histogram(values, nbins=100, fit_range=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if fit_range is not None:
        xmin, xmax = fit_range
        values = values[(values >= xmin) & (values <= xmax)]

    if len(values) < 20:
        return None

    counts, edges = np.histogram(values, bins=nbins, range=fit_range)
    centers = 0.5 * (edges[:-1] + edges[1:])

    mask = counts > 0
    xfit = centers[mask]
    yfit = counts[mask]

    if len(xfit) < 5:
        return None

    amp0 = np.max(yfit)
    mpv0 = xfit[np.argmax(yfit)]
    sigma0 = max(np.std(values) / 5.0, 20.0)

    try:
        popt, pcov = curve_fit(
            landau_like_pdf,
            xfit,
            yfit,
            p0=[amp0, mpv0, sigma0],
            bounds=([0.0, 0.0, 1e-6], [np.inf, np.inf, np.inf]),
            maxfev=20000
        )
        perr = np.sqrt(np.diag(pcov))
    except Exception:
        return None

    return {
        "counts": counts,
        "edges": edges,
        "centers": centers,
        "params": popt,
        "errors": perr,
        "mpv": popt[1],
        "mpv_err": perr[1],
    }


# ============================================================
# UTILITÀ RUN/HV
# ============================================================

def extract_hv_from_filename(filename):
    match = re.search(r'Bottom(\d{3,5})V', filename)
    if match:
        return int(match.group(1))
    return np.nan


def open_h5_or_gz(file_path):
    if file_path.endswith(".h5"):
        return file_path, None

    if file_path.endswith(".h5.gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False)
        tmp.close()
        with gzip.open(file_path, "rb") as fin, open(tmp.name, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        return tmp.name, tmp.name

    raise ValueError(f"Formato file non supportato: {file_path}")


# ============================================================
# CONFIG
# ============================================================

data_dir = os.path.expanduser("~/Downloads/test_071526")

run_files = sorted(glob.glob(os.path.join(
    data_dir,
    "MV_PMT_study_TopTriggExt_Bottom*V_offset40_0000_2.5Gs_40PT.h5.gz"
)))

# Esempi specifici, se vuoi limitare i run manualmente:
# run_files = [
#     os.path.join(data_dir, "MV_PMT_study_TopTriggExt_Bottom1600V_offset40_0000_2.5Gs_40PT.h5.gz"),
#     os.path.join(data_dir, "MV_PMT_study_TopTriggExt_Bottom1400V_offset40_0000_2.5Gs_40PT.h5.gz"),
#     os.path.join(data_dir, "MV_PMT_study_TopTriggExt_Bottom1200V_offset40_0000_2.5Gs_40PT.h5.gz"),
#     os.path.join(data_dir, "MV_PMT_study_TopTriggExt_Bottom1000V_offset40_0000_2.5Gs_40PT.h5.gz"),
# ]

# Evento e canali da plottare
EVENT_TO_PLOT = 130
CHANNELS_TO_PLOT = [16]        # NUMERI di canale, non posizioni: [16], [8, 9], ...

# Limite eventi analizzati; None = tutti
MAX_EVENTS_FOR_ANALYSIS = None

# Finestra integrazione [ns]
INTEGRATION_WINDOW_NS = (50.0, 150.0)

# Finestra baseline [ns]; se None usa tutti i campioni prima di tmin
BASELINE_WINDOW_NS = (0.0, 15.0)

# Soglia negativa [mV]: l'integrale viene fatto solo se il minimo nella finestra è <= soglia
NEGATIVE_THRESHOLD_MV = -5.0

Q = 60
MAX_NOTCH = 6
AUTO_DETECT = True

RF_BANDS = [
    (80e6, 95e6),
    (120e6, 135e6),
    (180e6, 200e6),
    (240e6, 260e6),
]

SG_WINDOW = 9
SG_ORDER = 2

ADC_TO_MV = 1000.0 / 4096.0
# NOMINAL_CHANNELS non serve piu': i canali li dichiara il file

# Parametri istogramma/fit
HIST_BINS = 100
FIT_RANGE = (400, 5000)
HIST_XLIM = (400, 5000)

# File output
RESULTS_CSV = "charge_results_by_event.csv"
MPV_CSV = "mpv_landau_fit_results.csv"


# ============================================================
# CHECK FILE
# ============================================================

if len(run_files) == 0:
    raise FileNotFoundError(
        f"Nessun file trovato in {data_dir} con pattern:\n"
        "MV_PMT_study_TopTriggExt_Bottom*V_offset40_0000_2.5Gs_40PT.h5.gz"
    )

missing = [p for p in run_files if not os.path.exists(p)]

if missing:
    raise FileNotFoundError("File mancanti:\n" + "\n".join(missing))

print("Run trovati:")
for f in run_files:
    print("  ", os.path.basename(f))


# ============================================================
# ANALISI MULTI-RUN
# ============================================================

all_results = []
plot_cache = []

for original_file_path in run_files:
    base_name = os.path.basename(original_file_path)
    run_label = base_name.replace(".h5.gz", "").replace(".h5", "")
    hv_value = extract_hv_from_filename(base_name)

    print(f"\n=== Analizzo {run_label} | HV = {hv_value} V ===")

    file_path, temp_file_to_remove = open_h5_or_gz(original_file_path)

    try:
        with apri_run(file_path) as f:
            config = f["/config"]
            sampling_time = float(config.attrs["SamplingTime"])
            fs = 1.0 / sampling_time

            event_keys, leggi_evento = eventi_di(f)
            canali_file, idx_canali = canali_di(f, CHANNELS_TO_PLOT)

            if MAX_EVENTS_FOR_ANALYSIS is not None:
                event_keys = event_keys[:MAX_EVENTS_FOR_ANALYSIS]

            first = leggi_evento(event_keys[0])
            total_samples = len(first)
            samples_per_channel = total_samples // len(canali_file)
            time_axis = np.arange(samples_per_channel) * sampling_time * 1e9

            for ev_index, ev_name in enumerate(event_keys):
                data_corr = leggi_evento(ev_name)

                if len(data_corr) != samples_per_channel * len(canali_file):
                    print(f"⚠️ Evento corrotto, skip: {ev_name}")
                    continue

                data_mat = data_corr.reshape((len(canali_file), samples_per_channel))

                for ch, i_ch in zip(CHANNELS_TO_PLOT, idx_canali):
                    wf_mv = data_mat[i_ch] * ADC_TO_MV

                    wf_filt = preprocess_waveform(
                        wf_mv, fs,
                        auto_detect=AUTO_DETECT,
                        max_notch=MAX_NOTCH,
                        q=Q,
                        rf_bands=RF_BANDS,
                        sg_window=SG_WINDOW,
                        sg_order=SG_ORDER,
                    )

                    baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window = compute_baseline_and_integral(
                        time_axis,
                        wf_filt,
                        INTEGRATION_WINDOW_NS[0],
                        INTEGRATION_WINDOW_NS[1],
                        baseline_window_ns=BASELINE_WINDOW_NS,
                        negative_threshold_mv=NEGATIVE_THRESHOLD_MV,
                    )

                    all_results.append({
                        "run": run_label,
                        "hv_V": hv_value,
                        "event": ev_index,
                        "channel": ch,
                        "baseline_mV": baseline,
                        "min_in_window_mV": min_in_window,
                        "passed_threshold": passed_threshold,
                        "integral_mV_ns": integral_mv_ns,
                    })

                    if ev_index == EVENT_TO_PLOT and ch in CHANNELS_TO_PLOT:
                        plot_cache.append({
                            "run": run_label,
                            "hv_V": hv_value,
                            "event": ev_index,
                            "channel": ch,
                            "time_ns": time_axis.copy(),
                            "wf_filt": wf_filt.copy(),
                            "wf_corr": wf_corr.copy(),
                            "signal_mask": signal_mask.copy(),
                            "baseline_mask": baseline_mask.copy(),
                            "integral_mV_ns": integral_mv_ns,
                            "passed_threshold": passed_threshold,
                            "min_in_window_mV": min_in_window,
                        })
    finally:
        if temp_file_to_remove is not None and os.path.exists(temp_file_to_remove):
            os.remove(temp_file_to_remove)


# ============================================================
# TABELLE RISULTATI
# ============================================================

df = pd.DataFrame(all_results)

summary = (
    df.groupby(["run", "hv_V", "channel"])
      .agg(
          n_entries=("integral_mV_ns", "count"),
          n_over_threshold=("passed_threshold", "sum"),
          mean_integral_mV_ns=("integral_mV_ns", "mean"),
          std_integral_mV_ns=("integral_mV_ns", "std"),
          min_integral_mV_ns=("integral_mV_ns", "min"),
          max_integral_mV_ns=("integral_mV_ns", "max"),
      )
      .sort_index()
)

display(summary)
display(df.head())

df.to_csv(RESULTS_CSV, index=False)
print(f"Salvato file risultati evento-per-evento: {RESULTS_CSV}")


# ============================================================
# PLOT DEI SEGNALI
# ============================================================

nplots = len(plot_cache)
if nplots == 0:
    print("Nessun segnale selezionato per il plot.")
else:
    fig, axes = plt.subplots(nplots, 1, figsize=(12, 3.2 * nplots), sharex=True)

    if nplots == 1:
        axes = [axes]

    for ax, item in zip(axes, plot_cache):
        ax.plot(item["time_ns"], item["wf_corr"], lw=1.0, color="crimson", alpha=0.9, label=f"Segnale Ch{item['channel']}")
        ax.axvspan(INTEGRATION_WINDOW_NS[0], INTEGRATION_WINDOW_NS[1], color="orange", alpha=0.22, label="Finestra integrazione")

        if np.any(item["baseline_mask"]):
            bt = item["time_ns"][item["baseline_mask"]]
            ax.axvspan(bt[0], bt[-1], color="green", alpha=0.12, label="Finestra baseline")

        ax.axhline(0.0, color="black", lw=0.8, ls="--")
        ax.axhline(NEGATIVE_THRESHOLD_MV, color="navy", lw=1.0, ls=":", label=f"Soglia Ch{item['channel']} = {NEGATIVE_THRESHOLD_MV:.1f} mV")

        status = "PASS" if item["passed_threshold"] else "FAIL"
        ax.set_ylabel(f"Ch {item['channel']} [mV]")
        ax.set_title(
            f"{item['run']} | HV = {item['hv_V']} V | Event {item['event']} | Ch {item['channel']} | "
            f"Carica = {item['integral_mV_ns']:.2f} mV·ns | "
            f"Min = {item['min_in_window_mV']:.2f} mV | {status}"
        )
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right", fontsize=8)

    axes[-1].set_xlabel("Time [ns]")
    plt.tight_layout()
    plt.show()


# ============================================================
# ISTOGRAMMI DEGLI INTEGRALI + FIT LANDAU-LIKE
# ============================================================

runs = sorted(df["run"].unique(), reverse=True)
mpv_results = []

for run in runs:
    run_df = df[df["run"] == run]
    hv_value = run_df["hv_V"].iloc[0]

    fig, axes = plt.subplots(len(CHANNELS_TO_PLOT), 1, figsize=(10, 3.4 * len(CHANNELS_TO_PLOT)), sharex=True)

    if len(CHANNELS_TO_PLOT) == 1:
        axes = [axes]

    for ax, ch in zip(axes, CHANNELS_TO_PLOT):
        vals = run_df.loc[
            (run_df["channel"] == ch) &
            (run_df["passed_threshold"]),
            "integral_mV_ns"
        ].values

        vals = vals[np.isfinite(vals)]
        vals = vals[vals > 0]

        n_pass = len(vals)

        ax.hist(
            vals,
            bins=HIST_BINS,
            range=FIT_RANGE,
            histtype="stepfilled",
            alpha=0.45,
            color="skyblue",
            label=f"Distribuzione carica Ch {ch}"
        )

        fitres = fit_landau_histogram(vals, nbins=HIST_BINS, fit_range=FIT_RANGE)

        if fitres is not None:
            xplot = np.linspace(FIT_RANGE[0], FIT_RANGE[1], 1200)
            yplot = landau_like_pdf(xplot, *fitres["params"])

            ax.plot(
                xplot,
                yplot,
                color="red",
                lw=2.0,
                label=f"Fit Landau-like | MPV = {fitres['mpv']:.1f} ± {fitres['mpv_err']:.1f} mV·ns"
            )

            mpv_results.append({
                "run": run,
                "hv_V": hv_value,
                "channel": ch,
                "mpv_mV_ns": fitres["mpv"],
                "mpv_err_mV_ns": fitres["mpv_err"],
                "n_entries_fit": n_pass,
            })

            title_suffix = f"MPV = {fitres['mpv']:.1f} ± {fitres['mpv_err']:.1f} mV·ns"
        else:
            mpv_results.append({
                "run": run,
                "hv_V": hv_value,
                "channel": ch,
                "mpv_mV_ns": np.nan,
                "mpv_err_mV_ns": np.nan,
                "n_entries_fit": n_pass,
            })

            title_suffix = "Fit non riuscito"

        ax.set_ylabel(f"Ch {ch}")
        ax.set_title(
            f"{run} | HV = {hv_value} V | Distribuzione carica Canale {ch} | "
            f"entries over threshold = {n_pass} | {title_suffix}"
        )
        ax.set_xlim(HIST_XLIM[0], HIST_XLIM[1])
        ax.grid(True, alpha=0.25)
        ax.legend(loc="upper right", fontsize=9)

    axes[-1].set_xlabel("Carica integrata nella finestra [mV·ns]")
    plt.tight_layout()
    plt.show()


# ============================================================
# TABELLA MPV
# ============================================================

df_mpv = pd.DataFrame(mpv_results)
display(df_mpv)

df_mpv.to_csv(MPV_CSV, index=False)
print(f"Salvato file MPV: {MPV_CSV}")


# ============================================================
# PLOT MPV vs HV
# ============================================================

df_mpv_valid = df_mpv.dropna(subset=["hv_V", "mpv_mV_ns"]).copy()

if len(df_mpv_valid) == 0:
    print("Nessun punto valido per il plot MPV vs HV.")
else:
    fig, ax = plt.subplots(figsize=(8, 5.5))

    for ch in sorted(df_mpv_valid["channel"].unique()):
        sel = df_mpv_valid[df_mpv_valid["channel"] == ch].sort_values("hv_V")

        ax.errorbar(
            sel["hv_V"],
            sel["mpv_mV_ns"],
            yerr=sel["mpv_err_mV_ns"],
            marker="o",
            linestyle="-",
            capsize=4,
            linewidth=1.5,
            label=f"Canale {ch}"
        )

        for _, row in sel.iterrows():
            ax.annotate(
                f"Ch{int(row['channel'])}",
                (row["hv_V"], row["mpv_mV_ns"]),
                textcoords="offset points",
                xytext=(5, 5),
                fontsize=8
            )

    ax.set_xlabel("HV [V]")
    ax.set_ylabel("MPV [mV·ns]")
    ax.set_title("MPV in funzione della HV")
    ax.grid(True, alpha=0.3)
    ax.legend()
    # ax.set_yscale("log")
    plt.tight_layout()
    plt.show()


In [ ]:
## === Import principali ===
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import h5py

# ============================================================
# LETTURA DEI FILE DELLA DAQ
#
# I file hanno due formati: v1 un dataset per evento (event0, event1, ...),
# v2 un unico dataset /events/waveforms -- nato per poter leggere mentre la
# run e' in corso, che con un dataset per evento non si puo' fare. E i canali
# registrati NON sono sempre 32: si prendono dal file, non si assumono.
# ============================================================
import contextlib, gzip, shutil, tempfile


@contextlib.contextmanager
def apri_run(path):
    """Apre un file della DAQ, compresso o no. A run finita diventa .h5.gz."""
    tmp = None
    if path.endswith(".gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False).name
        with gzip.open(path, "rb") as fin, open(tmp, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        path = tmp
    f = h5py.File(path, "r")
    try:
        yield f
    finally:
        f.close()
        if tmp:
            os.unlink(tmp)


def eventi_di(f):
    """(chiavi degli eventi, funzione che ne legge uno) per i due formati."""
    g = f["/events"]
    if "waveforms" in g:
        ds = g["waveforms"]
        return list(range(ds.shape[0])), lambda k: np.asarray(ds[k], dtype=np.float64)
    chiavi = sorted(g.keys(), key=lambda x: int(x.replace("event", "")))
    return chiavi, lambda k: np.asarray(g[k][:], dtype=np.float64)


def canali_di(f, voluti):
    """(canali nel file, indici di quelli chiesti).

    I canali si chiedono per NUMERO -- 16 e' il canale 16 del V1742 -- e non
    per posizione nella matrice: la posizione dipende da quali canali sono
    stati registrati in quella run, e cambia da una run all'altra.
    """
    lista = [int(c) for c in f["/config"].attrs["ChannelList"]]
    manca = [c for c in voluti if c not in lista]
    if manca:
        raise ValueError("Canali non nel file: %s. Il file contiene %s." % (manca, lista))
    return lista, [lista.index(c) for c in voluti]

import pandas as pd
from scipy.signal import iirnotch, filtfilt, find_peaks, butter, savgol_filter


# ============================================================
# RUMORE PERIODICO — FUNZIONI
# ============================================================

def detect_noise_frequencies(wf, fs, n_peaks=6, fmin=5e6, fmax=500e6):
    W = np.fft.rfft(wf)
    freq = np.fft.rfftfreq(len(wf), d=1/fs)
    spec = np.abs(W)

    mask = (freq > fmin) & (freq < fmax)
    freq = freq[mask]
    spec = spec[mask]

    peaks, props = find_peaks(spec, height=np.max(spec) * 0.2, distance=30)

    if len(peaks) == 0:
        return []

    peak_freqs = freq[peaks]
    order = np.argsort(props["peak_heights"])[::-1]
    peak_freqs = peak_freqs[order]

    return peak_freqs[:n_peaks]


def bandstop(wf, fs, f1, f2, order=3):
    b, a = butter(order, [f1/(fs/2), f2/(fs/2)], btype='bandstop')
    return filtfilt(b, a, wf)


def preprocess_waveform(wf_mv, fs, auto_detect=True, max_notch=6, q=60,
                        rf_bands=None, sg_window=9, sg_order=2):
    if rf_bands is None:
        rf_bands = []

    wf_filt = wf_mv.copy()

    if auto_detect:
        freqs = detect_noise_frequencies(wf_mv, fs, n_peaks=max_notch)
    else:
        freqs = []

    for f0 in freqs:
        if f0 < fs / 2:
            b, a = iirnotch(f0, q, fs)
            wf_filt = filtfilt(b, a, wf_filt)

    for f1, f2 in rf_bands:
        if f2 < fs / 2:
            wf_filt = bandstop(wf_filt, fs, f1, f2)

    if sg_window >= 3 and sg_window % 2 == 1 and sg_window < len(wf_filt):
        wf_filt = savgol_filter(wf_filt, window_length=sg_window, polyorder=sg_order)

    return wf_filt


def compute_baseline_and_integral(time_ns, wf_mv, tmin_ns, tmax_ns,
                                  baseline_window_ns=None,
                                  negative_threshold_mv=-5.0):
    signal_mask = (time_ns >= tmin_ns) & (time_ns <= tmax_ns)
    if not np.any(signal_mask):
        raise ValueError(f"Finestra di integrazione vuota: [{tmin_ns}, {tmax_ns}] ns")

    if baseline_window_ns is None:
        baseline_mask = time_ns < tmin_ns
    else:
        b0, b1 = baseline_window_ns
        baseline_mask = (time_ns >= b0) & (time_ns <= b1)

    baseline = np.mean(wf_mv[baseline_mask]) if np.any(baseline_mask) else 0.0
    wf_corr = wf_mv - baseline

    wf_in_window = wf_corr[signal_mask]
    min_in_window = np.min(wf_in_window)
    passed_threshold = min_in_window <= negative_threshold_mv

    if passed_threshold:
        integral_mv_ns = -np.trapezoid(wf_in_window, time_ns[signal_mask])
    else:
        integral_mv_ns = 0.0

    return baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window


# ============================================================
# UTILITÀ RUN/HV
# ============================================================

def extract_hv_from_filename(filename):
    match = re.search(r'_(\d{3,5})V(?:_|\.|$)', filename)
    if match:
        return int(match.group(1))
    return np.nan


def integral_mVns_to_charge_C(integral_mV_ns, load_ohm=50.0):
    return integral_mV_ns * 1e-12 / load_ohm


# ============================================================
# CONFIG
# ============================================================

data_dir = os.path.expanduser("~/Downloads")

run_files = [
    "MV_PMT_study_test_1600V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1400V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1200V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1000V_offset40_0000_2.5Gs_40PT.h5",
]

CHANNELS_TO_PLOT = [16]        # NUMERI di canale, non posizioni: [16], [8, 9], ...

# Limite eventi analizzati; None = tutti
MAX_EVENTS_FOR_ANALYSIS = None

# Finestra integrazione [ns]
INTEGRATION_WINDOW_NS = (50.0, 150.0)

# Finestra baseline [ns]
BASELINE_WINDOW_NS = (0.0, 15.0)

# Soglia negativa [mV]
NEGATIVE_THRESHOLD_MV = -5.0

Q = 60
MAX_NOTCH = 6
AUTO_DETECT = True

RF_BANDS = [
    (80e6, 95e6),
    (120e6, 135e6),
    (180e6, 200e6),
    (240e6, 260e6),
]

SG_WINDOW = 9
SG_ORDER = 2

ADC_TO_MV = 1000.0 / 4096.0
# NOMINAL_CHANNELS non serve piu': i canali li dichiara il file
LOAD_OHM = 50.0

RESULTS_CSV = "anode_current_event_by_event.csv"
SUMMARY_CSV = "anode_current_summary.csv"


# ============================================================
# CHECK FILE
# ============================================================

file_paths = [os.path.join(data_dir, name) for name in run_files]
missing = [p for p in file_paths if not os.path.exists(p)]

if missing:
    raise FileNotFoundError("File mancanti:\n" + "\n".join(missing))


# ============================================================
# ANALISI MULTI-RUN
# ============================================================

all_results = []

for file_path in file_paths:
    base_name = os.path.basename(file_path)
    run_label = base_name.replace(".h5", "")
    hv_value = extract_hv_from_filename(base_name)

    print(f"\n=== Analizzo {run_label} | HV = {hv_value} V ===")

    with apri_run(file_path) as f:
        config = f["/config"]
        sampling_time = float(config.attrs["SamplingTime"])   # s
        fs = 1.0 / sampling_time

        event_keys, leggi_evento = eventi_di(f)
        canali_file, idx_canali = canali_di(f, CHANNELS_TO_PLOT)

        if MAX_EVENTS_FOR_ANALYSIS is not None:
            event_keys = event_keys[:MAX_EVENTS_FOR_ANALYSIS]

        first = leggi_evento(event_keys[0])
        total_samples = len(first)
        samples_per_channel = total_samples // len(canali_file)
        event_duration_s = samples_per_channel * sampling_time
        event_rate_Hz = 1.0 / event_duration_s

        time_axis = np.arange(samples_per_channel) * sampling_time * 1e9

        print(f"Sampling time = {sampling_time*1e9:.3f} ns")
        print(f"Samples/channel = {samples_per_channel}")
        print(f"Durata evento = {event_duration_s*1e9:.3f} ns")
        print(f"Rate equivalente = {event_rate_Hz/1e6:.3f} MHz")

        for ev_index, ev_name in enumerate(event_keys):
            data_corr = leggi_evento(ev_name)

            if len(data_corr) != samples_per_channel * len(canali_file):
                print(f"⚠️ Evento corrotto, skip: {ev_name}")
                continue

            data_mat = data_corr.reshape((len(canali_file), samples_per_channel))

            for ch, i_ch in zip(CHANNELS_TO_PLOT, idx_canali):
                wf_mv = data_mat[i_ch] * ADC_TO_MV

                wf_filt = preprocess_waveform(
                    wf_mv, fs,
                    auto_detect=AUTO_DETECT,
                    max_notch=MAX_NOTCH,
                    q=Q,
                    rf_bands=RF_BANDS,
                    sg_window=SG_WINDOW,
                    sg_order=SG_ORDER,
                )

                baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window = compute_baseline_and_integral(
                    time_axis,
                    wf_filt,
                    INTEGRATION_WINDOW_NS[0],
                    INTEGRATION_WINDOW_NS[1],
                    baseline_window_ns=BASELINE_WINDOW_NS,
                    negative_threshold_mv=NEGATIVE_THRESHOLD_MV,
                )

                charge_C = integral_mVns_to_charge_C(integral_mv_ns, LOAD_OHM)
                mean_anode_current_A = charge_C * event_rate_Hz

                all_results.append({
                    "run": run_label,
                    "hv_V": hv_value,
                    "event": ev_index,
                    "channel": ch,
                    "sampling_time_s": sampling_time,
                    "samples_per_channel": samples_per_channel,
                    "event_duration_s": event_duration_s,
                    "event_rate_Hz": event_rate_Hz,
                    "baseline_mV": baseline,
                    "min_in_window_mV": min_in_window,
                    "passed_threshold": passed_threshold,
                    "integral_mV_ns": integral_mv_ns,
                    "charge_C": charge_C,
                    "mean_anode_current_A": mean_anode_current_A,
                    "mean_anode_current_nA": mean_anode_current_A * 1e9,
                    "mean_anode_current_uA": mean_anode_current_A * 1e6,
                })


# ============================================================
# TABELLE RISULTATI
# ============================================================

df = pd.DataFrame(all_results)

display(df.head())

summary = (
    df.groupby(["run", "hv_V", "channel"])
      .agg(
          n_events=("event", "count"),
          n_over_threshold=("passed_threshold", "sum"),
          event_rate_Hz=("event_rate_Hz", "first"),
          mean_charge_C=("charge_C", "mean"),
          mean_charge_pC=("charge_C", lambda x: np.mean(x) * 1e12),
          mean_charge_over_thr_pC=("charge_C", lambda x: np.mean(x[x > 0]) * 1e12 if np.any(x > 0) else np.nan),
          mean_I_anode_A=("mean_anode_current_A", "mean"),
          mean_I_anode_nA=("mean_anode_current_nA", "mean"),
          mean_I_anode_uA=("mean_anode_current_uA", "mean"),
          mean_I_anode_over_thr_nA=("mean_anode_current_nA", lambda x: np.mean(x[x > 0]) if np.any(x > 0) else np.nan),
          median_integral_mV_ns=("integral_mV_ns", "median"),
          mean_integral_mV_ns=("integral_mV_ns", "mean"),
      )
      .sort_values(["hv_V", "channel"])
      .reset_index()
)

display(summary)

df.to_csv(RESULTS_CSV, index=False)
summary.to_csv(SUMMARY_CSV, index=False)

print(f"Salvato file evento-per-evento: {RESULTS_CSV}")
print(f"Salvato file riassunto: {SUMMARY_CSV}")


# ============================================================
# PLOT CORRENTE MEDIA DI ANODO vs HV
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5.5))

for ch in sorted(summary["channel"].unique()):
    sel = summary[summary["channel"] == ch].sort_values("hv_V")

    ax.plot(
        sel["hv_V"],
        sel["mean_I_anode_nA"],
        marker="o",
        linewidth=1.8,
        label=f"Canale {ch} | media su tutti gli eventi"
    )

    ax.plot(
        sel["hv_V"],
        sel["mean_I_anode_over_thr_nA"],
        marker="s",
        linestyle="--",
        linewidth=1.5,
        label=f"Canale {ch} | media eventi over threshold"
    )

ax.set_xlabel("HV [V]")
ax.set_ylabel("Corrente media di anodo [nA]")
ax.set_title("Corrente media di anodo in funzione della HV")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


# ============================================================
# PLOT CARICA MEDIA vs HV
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5.5))

for ch in sorted(summary["channel"].unique()):
    sel = summary[summary["channel"] == ch].sort_values("hv_V")

    ax.plot(
        sel["hv_V"],
        sel["mean_charge_pC"],
        marker="o",
        linewidth=1.8,
        label=f"Canale {ch} | media su tutti gli eventi"
    )

    ax.plot(
        sel["hv_V"],
        sel["mean_charge_over_thr_pC"],
        marker="s",
        linestyle="--",
        linewidth=1.5,
        label=f"Canale {ch} | media eventi over threshold"
    )

ax.set_xlabel("HV [V]")
ax.set_ylabel("Carica media per evento [pC]")
ax.set_title("Carica media per evento in funzione della HV")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
## === Import principali ===
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import h5py

# ============================================================
# LETTURA DEI FILE DELLA DAQ
#
# I file hanno due formati: v1 un dataset per evento (event0, event1, ...),
# v2 un unico dataset /events/waveforms -- nato per poter leggere mentre la
# run e' in corso, che con un dataset per evento non si puo' fare. E i canali
# registrati NON sono sempre 32: si prendono dal file, non si assumono.
# ============================================================
import contextlib, gzip, shutil, tempfile


@contextlib.contextmanager
def apri_run(path):
    """Apre un file della DAQ, compresso o no. A run finita diventa .h5.gz."""
    tmp = None
    if path.endswith(".gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False).name
        with gzip.open(path, "rb") as fin, open(tmp, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        path = tmp
    f = h5py.File(path, "r")
    try:
        yield f
    finally:
        f.close()
        if tmp:
            os.unlink(tmp)


def eventi_di(f):
    """(chiavi degli eventi, funzione che ne legge uno) per i due formati."""
    g = f["/events"]
    if "waveforms" in g:
        ds = g["waveforms"]
        return list(range(ds.shape[0])), lambda k: np.asarray(ds[k], dtype=np.float64)
    chiavi = sorted(g.keys(), key=lambda x: int(x.replace("event", "")))
    return chiavi, lambda k: np.asarray(g[k][:], dtype=np.float64)


def canali_di(f, voluti):
    """(canali nel file, indici di quelli chiesti).

    I canali si chiedono per NUMERO -- 16 e' il canale 16 del V1742 -- e non
    per posizione nella matrice: la posizione dipende da quali canali sono
    stati registrati in quella run, e cambia da una run all'altra.
    """
    lista = [int(c) for c in f["/config"].attrs["ChannelList"]]
    manca = [c for c in voluti if c not in lista]
    if manca:
        raise ValueError("Canali non nel file: %s. Il file contiene %s." % (manca, lista))
    return lista, [lista.index(c) for c in voluti]

import pandas as pd
from scipy.signal import iirnotch, filtfilt, find_peaks, butter, savgol_filter


# ============================================================
# RUMORE PERIODICO — FUNZIONI
# ============================================================

def detect_noise_frequencies(wf, fs, n_peaks=6, fmin=5e6, fmax=500e6):
    W = np.fft.rfft(wf)
    freq = np.fft.rfftfreq(len(wf), d=1/fs)
    spec = np.abs(W)

    mask = (freq > fmin) & (freq < fmax)
    freq = freq[mask]
    spec = spec[mask]

    peaks, props = find_peaks(spec, height=np.max(spec) * 0.2, distance=30)

    if len(peaks) == 0:
        return []

    peak_freqs = freq[peaks]
    order = np.argsort(props["peak_heights"])[::-1]
    peak_freqs = peak_freqs[order]

    return peak_freqs[:n_peaks]


def bandstop(wf, fs, f1, f2, order=3):
    b, a = butter(order, [f1/(fs/2), f2/(fs/2)], btype='bandstop')
    return filtfilt(b, a, wf)


def preprocess_waveform(wf_mv, fs, auto_detect=True, max_notch=6, q=60,
                        rf_bands=None, sg_window=9, sg_order=2):
    if rf_bands is None:
        rf_bands = []

    wf_filt = wf_mv.copy()

    if auto_detect:
        freqs = detect_noise_frequencies(wf_mv, fs, n_peaks=max_notch)
    else:
        freqs = []

    for f0 in freqs:
        if f0 < fs / 2:
            b, a = iirnotch(f0, q, fs)
            wf_filt = filtfilt(b, a, wf_filt)

    for f1, f2 in rf_bands:
        if f2 < fs / 2:
            wf_filt = bandstop(wf_filt, fs, f1, f2)

    if sg_window >= 3 and sg_window % 2 == 1 and sg_window < len(wf_filt):
        wf_filt = savgol_filter(wf_filt, window_length=sg_window, polyorder=sg_order)

    return wf_filt


def compute_baseline_and_integral(time_ns, wf_mv, tmin_ns, tmax_ns,
                                  baseline_window_ns=None,
                                  negative_threshold_mv=-5.0):
    signal_mask = (time_ns >= tmin_ns) & (time_ns <= tmax_ns)
    if not np.any(signal_mask):
        raise ValueError(f"Finestra di integrazione vuota: [{tmin_ns}, {tmax_ns}] ns")

    if baseline_window_ns is None:
        baseline_mask = time_ns < tmin_ns
    else:
        b0, b1 = baseline_window_ns
        baseline_mask = (time_ns >= b0) & (time_ns <= b1)

    baseline = np.mean(wf_mv[baseline_mask]) if np.any(baseline_mask) else 0.0
    wf_corr = wf_mv - baseline

    wf_in_window = wf_corr[signal_mask]
    min_in_window = np.min(wf_in_window)
    passed_threshold = min_in_window <= negative_threshold_mv

    if passed_threshold:
        integral_mv_ns = -np.trapezoid(wf_in_window, time_ns[signal_mask])
    else:
        integral_mv_ns = 0.0

    return baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window


# ============================================================
# UTILITÀ RUN/HV
# ============================================================

def extract_hv_from_filename(filename):
    match = re.search(r'_(\d{3,5})V(?:_|\.|$)', filename)
    if match:
        return int(match.group(1))
    return np.nan


# ============================================================
# CONFIG
# ============================================================

data_dir = os.path.expanduser("~/Downloads")

run_files = [
    "MV_PMT_study_test_1600V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1400V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1200V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1000V_offset40_0000_2.5Gs_40PT.h5",
]

CHANNELS_TO_PLOT = [16]        # NUMERI di canale, non posizioni: [16], [8, 9], ...

# Limite eventi analizzati; None = tutti
MAX_EVENTS_FOR_ANALYSIS = None

# Finestra integrazione [ns]
INTEGRATION_WINDOW_NS = (50.0, 150.0)

# Finestra baseline [ns]
BASELINE_WINDOW_NS = (0.0, 15.0)

# Soglia negativa [mV]
NEGATIVE_THRESHOLD_MV = -5.0

Q = 60
MAX_NOTCH = 6
AUTO_DETECT = True

RF_BANDS = [
    (80e6, 95e6),
    (120e6, 135e6),
    (180e6, 200e6),
    (240e6, 260e6),
]

SG_WINDOW = 9
SG_ORDER = 2

ADC_TO_MV = 1000.0 / 4096.0
# NOMINAL_CHANNELS non serve piu': i canali li dichiara il file
LOAD_OHM = 50.0

RESULTS_CSV = "anode_current_event_by_event.csv"
SUMMARY_CSV = "anode_current_summary.csv"


# ============================================================
# CHECK FILE
# ============================================================

file_paths = [os.path.join(data_dir, name) for name in run_files]
missing = [p for p in file_paths if not os.path.exists(p)]

if missing:
    raise FileNotFoundError("File mancanti:\n" + "\n".join(missing))


# ============================================================
# ANALISI MULTI-RUN
# ============================================================

all_results = []

for file_path in file_paths:
    base_name = os.path.basename(file_path)
    run_label = base_name.replace(".h5", "")
    hv_value = extract_hv_from_filename(base_name)

    print(f"\n=== Analizzo {run_label} | HV = {hv_value} V ===")

    with apri_run(file_path) as f:
        config = f["/config"]
        sampling_time = float(config.attrs["SamplingTime"])   # s
        fs = 1.0 / sampling_time

        event_keys, leggi_evento = eventi_di(f)
        canali_file, idx_canali = canali_di(f, CHANNELS_TO_PLOT)

        if MAX_EVENTS_FOR_ANALYSIS is not None:
            event_keys = event_keys[:MAX_EVENTS_FOR_ANALYSIS]

        first = leggi_evento(event_keys[0])
        total_samples = len(first)
        samples_per_channel = total_samples // len(canali_file)
        event_duration_s = samples_per_channel * sampling_time
        event_rate_Hz = 1.0 / event_duration_s

        time_axis = np.arange(samples_per_channel) * sampling_time * 1e9

        print(f"Sampling time = {sampling_time*1e9:.3f} ns")
        print(f"Samples/channel = {samples_per_channel}")
        print(f"Durata evento = {event_duration_s*1e9:.3f} ns")
        print(f"Rate equivalente = {event_rate_Hz/1e6:.3f} MHz")

        for ev_index, ev_name in enumerate(event_keys):
            data_corr = leggi_evento(ev_name)

            if len(data_corr) != samples_per_channel * len(canali_file):
                print(f"⚠️ Evento corrotto, skip: {ev_name}")
                continue

            data_mat = data_corr.reshape((len(canali_file), samples_per_channel))

            for ch, i_ch in zip(CHANNELS_TO_PLOT, idx_canali):
                wf_mv = data_mat[i_ch] * ADC_TO_MV

                wf_filt = preprocess_waveform(
                    wf_mv, fs,
                    auto_detect=AUTO_DETECT,
                    max_notch=MAX_NOTCH,
                    q=Q,
                    rf_bands=RF_BANDS,
                    sg_window=SG_WINDOW,
                    sg_order=SG_ORDER,
                )

                baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window = compute_baseline_and_integral(
                    time_axis,
                    wf_filt,
                    INTEGRATION_WINDOW_NS[0],
                    INTEGRATION_WINDOW_NS[1],
                    baseline_window_ns=BASELINE_WINDOW_NS,
                    negative_threshold_mv=NEGATIVE_THRESHOLD_MV,
                )

                charge_C = integral_mv_ns * 1e-12 / LOAD_OHM
                mean_anode_current_A = charge_C * event_rate_Hz
                peak_current_A = (abs(np.min(wf_corr[signal_mask])) * 1e-3) / LOAD_OHM if np.any(signal_mask) else np.nan

                all_results.append({
                    "run": run_label,
                    "hv_V": hv_value,
                    "event": ev_index,
                    "channel": ch,
                    "sampling_time_s": sampling_time,
                    "samples_per_channel": samples_per_channel,
                    "event_duration_s": event_duration_s,
                    "event_rate_Hz": event_rate_Hz,
                    "baseline_mV": baseline,
                    "min_in_window_mV": min_in_window,
                    "passed_threshold": passed_threshold,
                    "integral_mV_ns": integral_mv_ns,
                    "charge_C": charge_C,
                    "mean_anode_current_A": mean_anode_current_A,
                    "mean_anode_current_nA": mean_anode_current_A * 1e9,
                    "mean_anode_current_uA": mean_anode_current_A * 1e6,
                    "peak_current_A": peak_current_A,
                    "peak_current_nA": peak_current_A * 1e9 if np.isfinite(peak_current_A) else np.nan,
                    "peak_current_uA": peak_current_A * 1e6 if np.isfinite(peak_current_A) else np.nan,
                })


# ============================================================
# TABELLE RISULTATI
# ============================================================

df = pd.DataFrame(all_results)

display(df.head())

summary = (
    df.groupby(["run", "hv_V", "channel"])
      .agg(
          n_events=("event", "count"),
          n_over_threshold=("passed_threshold", "sum"),
          event_rate_Hz=("event_rate_Hz", "first"),
          mean_charge_C=("charge_C", "mean"),
          mean_charge_pC=("charge_C", lambda x: np.mean(x) * 1e12),
          mean_charge_over_thr_pC=("charge_C", lambda x: np.mean(x[x > 0]) * 1e12 if np.any(x > 0) else np.nan),
          mean_I_anode_A=("mean_anode_current_A", "mean"),
          mean_I_anode_nA=("mean_anode_current_nA", "mean"),
          mean_I_anode_uA=("mean_anode_current_uA", "mean"),
          mean_I_anode_over_thr_nA=("mean_anode_current_nA", lambda x: np.mean(x[x > 0]) if np.any(x > 0) else np.nan),
          mean_peak_current_nA=("peak_current_nA", "mean"),
          mean_peak_current_uA=("peak_current_uA", "mean"),
          median_integral_mV_ns=("integral_mV_ns", "median"),
          mean_integral_mV_ns=("integral_mV_ns", "mean"),
      )
      .sort_values(["hv_V", "channel"])
      .reset_index()
)

display(summary)

df.to_csv(RESULTS_CSV, index=False)
summary.to_csv(SUMMARY_CSV, index=False)

print(f"Salvato file evento-per-evento: {RESULTS_CSV}")
print(f"Salvato file riassunto: {SUMMARY_CSV}")


# ============================================================
# PLOT CORRENTE MEDIA DI ANODO vs HV
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5.5))

for ch in sorted(summary["channel"].unique()):
    sel = summary[summary["channel"] == ch].sort_values("hv_V")

    ax.plot(
        sel["hv_V"],
        sel["mean_I_anode_nA"],
        marker="o",
        linewidth=1.8,
        label=f"Canale {ch} | media su tutti gli eventi"
    )

    ax.plot(
        sel["hv_V"],
        sel["mean_I_anode_over_thr_nA"],
        marker="s",
        linestyle="--",
        linewidth=1.5,
        label=f"Canale {ch} | media eventi over threshold"
    )

ax.set_xlabel("HV [V]")
ax.set_ylabel("Corrente media di anodo [nA]")
ax.set_title("Corrente media di anodo in funzione della HV")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


# ============================================================
# PLOT CORRENTE DI PICCO vs HV
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5.5))

for ch in sorted(summary["channel"].unique()):
    sel = summary[summary["channel"] == ch].sort_values("hv_V")

    ax.plot(
        sel["hv_V"],
        sel["mean_peak_current_nA"],
        marker="o",
        linewidth=1.8,
        label=f"Canale {ch}"
    )

ax.set_xlabel("HV [V]")
ax.set_ylabel("Corrente di picco di anodo [nA]")
ax.set_title("Corrente di picco di anodo in funzione della HV")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


# ============================================================
# PLOT CARICA MEDIA vs HV
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5.5))

for ch in sorted(summary["channel"].unique()):
    sel = summary[summary["channel"] == ch].sort_values("hv_V")

    ax.plot(
        sel["hv_V"],
        sel["mean_charge_pC"],
        marker="o",
        linewidth=1.8,
        label=f"Canale {ch} | media su tutti gli eventi"
    )

    ax.plot(
        sel["hv_V"],
        sel["mean_charge_over_thr_pC"],
        marker="s",
        linestyle="--",
        linewidth=1.5,
        label=f"Canale {ch} | media eventi over threshold"
    )

ax.set_xlabel("HV [V]")
ax.set_ylabel("Carica media per evento [pC]")
ax.set_title("Carica media per evento in funzione della HV")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
## === Import principali ===
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import h5py

# ============================================================
# LETTURA DEI FILE DELLA DAQ
#
# I file hanno due formati: v1 un dataset per evento (event0, event1, ...),
# v2 un unico dataset /events/waveforms -- nato per poter leggere mentre la
# run e' in corso, che con un dataset per evento non si puo' fare. E i canali
# registrati NON sono sempre 32: si prendono dal file, non si assumono.
# ============================================================
import contextlib, gzip, shutil, tempfile


@contextlib.contextmanager
def apri_run(path):
    """Apre un file della DAQ, compresso o no. A run finita diventa .h5.gz."""
    tmp = None
    if path.endswith(".gz"):
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False).name
        with gzip.open(path, "rb") as fin, open(tmp, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        path = tmp
    f = h5py.File(path, "r")
    try:
        yield f
    finally:
        f.close()
        if tmp:
            os.unlink(tmp)


def eventi_di(f):
    """(chiavi degli eventi, funzione che ne legge uno) per i due formati."""
    g = f["/events"]
    if "waveforms" in g:
        ds = g["waveforms"]
        return list(range(ds.shape[0])), lambda k: np.asarray(ds[k], dtype=np.float64)
    chiavi = sorted(g.keys(), key=lambda x: int(x.replace("event", "")))
    return chiavi, lambda k: np.asarray(g[k][:], dtype=np.float64)


def canali_di(f, voluti):
    """(canali nel file, indici di quelli chiesti).

    I canali si chiedono per NUMERO -- 16 e' il canale 16 del V1742 -- e non
    per posizione nella matrice: la posizione dipende da quali canali sono
    stati registrati in quella run, e cambia da una run all'altra.
    """
    lista = [int(c) for c in f["/config"].attrs["ChannelList"]]
    manca = [c for c in voluti if c not in lista]
    if manca:
        raise ValueError("Canali non nel file: %s. Il file contiene %s." % (manca, lista))
    return lista, [lista.index(c) for c in voluti]

import pandas as pd
from scipy.signal import iirnotch, filtfilt, find_peaks, butter, savgol_filter


# ============================================================
# RUMORE PERIODICO — FUNZIONI
# ============================================================

def detect_noise_frequencies(wf, fs, n_peaks=6, fmin=5e6, fmax=500e6):
    W = np.fft.rfft(wf)
    freq = np.fft.rfftfreq(len(wf), d=1/fs)
    spec = np.abs(W)

    mask = (freq > fmin) & (freq < fmax)
    freq = freq[mask]
    spec = spec[mask]

    peaks, props = find_peaks(spec, height=np.max(spec) * 0.2, distance=30)

    if len(peaks) == 0:
        return []

    peak_freqs = freq[peaks]
    order = np.argsort(props["peak_heights"])[::-1]
    peak_freqs = peak_freqs[order]

    return peak_freqs[:n_peaks]


def bandstop(wf, fs, f1, f2, order=3):
    b, a = butter(order, [f1/(fs/2), f2/(fs/2)], btype='bandstop')
    return filtfilt(b, a, wf)


def preprocess_waveform(wf_mv, fs, auto_detect=True, max_notch=6, q=60,
                        rf_bands=None, sg_window=9, sg_order=2):
    if rf_bands is None:
        rf_bands = []

    wf_filt = wf_mv.copy()

    if auto_detect:
        freqs = detect_noise_frequencies(wf_mv, fs, n_peaks=max_notch)
    else:
        freqs = []

    for f0 in freqs:
        if f0 < fs / 2:
            b, a = iirnotch(f0, q, fs)
            wf_filt = filtfilt(b, a, wf_filt)

    for f1, f2 in rf_bands:
        if f2 < fs / 2:
            wf_filt = bandstop(wf_filt, fs, f1, f2)

    if sg_window >= 3 and sg_window % 2 == 1 and sg_window < len(wf_filt):
        wf_filt = savgol_filter(wf_filt, window_length=sg_window, polyorder=sg_order)

    return wf_filt


def compute_baseline_and_integral(time_ns, wf_mv, tmin_ns, tmax_ns,
                                  baseline_window_ns=None,
                                  negative_threshold_mv=-5.0):
    signal_mask = (time_ns >= tmin_ns) & (time_ns <= tmax_ns)
    if not np.any(signal_mask):
        raise ValueError(f"Finestra di integrazione vuota: [{tmin_ns}, {tmax_ns}] ns")

    if baseline_window_ns is None:
        baseline_mask = time_ns < tmin_ns
    else:
        b0, b1 = baseline_window_ns
        baseline_mask = (time_ns >= b0) & (time_ns <= b1)

    baseline = np.mean(wf_mv[baseline_mask]) if np.any(baseline_mask) else 0.0
    wf_corr = wf_mv - baseline

    wf_in_window = wf_corr[signal_mask]
    min_in_window = np.min(wf_in_window)
    passed_threshold = min_in_window <= negative_threshold_mv

    if passed_threshold:
        integral_mv_ns = -np.trapezoid(wf_in_window, time_ns[signal_mask])
    else:
        integral_mv_ns = 0.0

    return baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window


# ============================================================
# UTILITÀ RUN/HV
# ============================================================

def extract_hv_from_filename(filename):
    match = re.search(r'_(\d{3,5})V(?:_|\.|$)', filename)
    if match:
        return int(match.group(1))
    return np.nan


# ============================================================
# DIAGNOSTICA SATURAZIONE
# ============================================================

DIVIDER_CURRENT_A = 0.67e-3         # E5859 datasheet: 0.67 mA
DC_LINEAR_OUTPUT_A = 33e-6          # E5859 datasheet: 33 uA @ 2700 V
HANDBOOK_1PCT_RATIO = 0.014         # PMT handbook: ~1.4% per 1% linearity
PRACTICAL_10X_RATIO = 0.10          # divider current at least 10x anode average current
PRACTICAL_20X_RATIO = 0.05          # divider current at least 20x anode average current

def classify_saturation(mean_I_A, peak_I_A=None):
    ratio = mean_I_A / DIVIDER_CURRENT_A if DIVIDER_CURRENT_A > 0 else np.nan

    flags = []

    if mean_I_A >= DC_LINEAR_OUTPUT_A:
        flags.append("mean_I >= datasheet_DC_limit")

    if ratio >= PRACTICAL_10X_RATIO:
        flags.append("mean_I >= 10% divider current")
    elif ratio >= PRACTICAL_20X_RATIO:
        flags.append("mean_I >= 5% divider current")

    if ratio >= HANDBOOK_1PCT_RATIO:
        flags.append("mean_I >= handbook_1pct_linearity_limit")

    if mean_I_A >= DC_LINEAR_OUTPUT_A or ratio >= PRACTICAL_10X_RATIO:
        status = "LIKELY SATURATION / NON-LINEAR"
    elif ratio >= PRACTICAL_20X_RATIO or ratio >= HANDBOOK_1PCT_RATIO:
        status = "ATTENTION / POSSIBLE NON-LINEARITY"
    else:
        status = "SAFE"

    note = "; ".join(flags) if flags else "within conservative divider-current criteria"

    return status, ratio, note


# ============================================================
# CONFIG
# ============================================================

data_dir = os.path.expanduser("~/Downloads")

run_files = [
    "MV_PMT_study_test_1600V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1400V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1200V_offset40_0000_2.5Gs_40PT.h5",
    "MV_PMT_study_test_1000V_offset40_0000_2.5Gs_40PT.h5",
]

CHANNELS_TO_ANALYZE = [16]        # NUMERI di canale, non posizioni: [16], [8, 9], ...
MAX_EVENTS_FOR_ANALYSIS = None

INTEGRATION_WINDOW_NS = (50.0, 150.0)
BASELINE_WINDOW_NS = (0.0, 15.0)
NEGATIVE_THRESHOLD_MV = -5.0

Q = 60
MAX_NOTCH = 6
AUTO_DETECT = True

RF_BANDS = [
    (80e6, 95e6),
    (120e6, 135e6),
    (180e6, 200e6),
    (240e6, 260e6),
]

SG_WINDOW = 9
SG_ORDER = 2

ADC_TO_MV = 1000.0 / 4096.0
# NOMINAL_CHANNELS non serve piu': i canali li dichiara il file
LOAD_OHM = 50.0

EVENT_CSV = "saturation_event_by_event.csv"
SUMMARY_CSV = "saturation_summary.csv"
DIAGNOSIS_CSV = "saturation_diagnosis.csv"


# ============================================================
# CHECK FILE
# ============================================================

file_paths = [os.path.join(data_dir, name) for name in run_files]
missing = [p for p in file_paths if not os.path.exists(p)]

if missing:
    raise FileNotFoundError("File mancanti:\n" + "\n".join(missing))


# ============================================================
# LOOP RUN
# ============================================================

all_results = []

for file_path in file_paths:
    base_name = os.path.basename(file_path)
    run_label = base_name.replace(".h5", "")
    hv_value = extract_hv_from_filename(base_name)

    print(f"\n=== Analizzo {run_label} | HV = {hv_value} V ===")

    with apri_run(file_path) as f:
        config = f["/config"]
        sampling_time = float(config.attrs["SamplingTime"])   # s
        fs = 1.0 / sampling_time

        event_keys, leggi_evento = eventi_di(f)
        canali_file, idx_canali = canali_di(f, CHANNELS_TO_ANALYZE)

        if MAX_EVENTS_FOR_ANALYSIS is not None:
            event_keys = event_keys[:MAX_EVENTS_FOR_ANALYSIS]

        first = leggi_evento(event_keys[0])
        total_samples = len(first)
        samples_per_channel = total_samples // len(canali_file)
        event_duration_s = samples_per_channel * sampling_time
        event_rate_Hz = 1.0 / event_duration_s

        time_axis = np.arange(samples_per_channel) * sampling_time * 1e9

        for ev_index, ev_name in enumerate(event_keys):
            data_corr = leggi_evento(ev_name)

            if len(data_corr) != samples_per_channel * len(canali_file):
                print(f"⚠️ Evento corrotto, skip: {ev_name}")
                continue

            data_mat = data_corr.reshape((len(canali_file), samples_per_channel))

            for ch, i_ch in zip(CHANNELS_TO_ANALYZE, idx_canali):
                wf_mv = data_mat[i_ch] * ADC_TO_MV

                wf_filt = preprocess_waveform(
                    wf_mv, fs,
                    auto_detect=AUTO_DETECT,
                    max_notch=MAX_NOTCH,
                    q=Q,
                    rf_bands=RF_BANDS,
                    sg_window=SG_WINDOW,
                    sg_order=SG_ORDER,
                )

                baseline, wf_corr, integral_mv_ns, signal_mask, baseline_mask, passed_threshold, min_in_window = compute_baseline_and_integral(
                    time_axis,
                    wf_filt,
                    INTEGRATION_WINDOW_NS[0],
                    INTEGRATION_WINDOW_NS[1],
                    baseline_window_ns=BASELINE_WINDOW_NS,
                    negative_threshold_mv=NEGATIVE_THRESHOLD_MV,
                )

                charge_C = integral_mv_ns * 1e-12 / LOAD_OHM
                mean_anode_current_A = charge_C * event_rate_Hz

                if np.any(signal_mask):
                    peak_current_A = abs(np.min(wf_corr[signal_mask])) * 1e-3 / LOAD_OHM
                else:
                    peak_current_A = np.nan

                all_results.append({
                    "run": run_label,
                    "hv_V": hv_value,
                    "event": ev_index,
                    "channel": ch,
                    "event_rate_Hz": event_rate_Hz,
                    "baseline_mV": baseline,
                    "min_in_window_mV": min_in_window,
                    "passed_threshold": passed_threshold,
                    "integral_mV_ns": integral_mv_ns,
                    "charge_C": charge_C,
                    "mean_anode_current_A": mean_anode_current_A,
                    "mean_anode_current_uA": mean_anode_current_A * 1e6,
                    "peak_current_A": peak_current_A,
                    "peak_current_mA": peak_current_A * 1e3 if np.isfinite(peak_current_A) else np.nan,
                })


# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(all_results)
df.to_csv(EVENT_CSV, index=False)

summary = (
    df.groupby(["run", "hv_V", "channel"])
      .agg(
          n_events=("event", "count"),
          n_over_threshold=("passed_threshold", "sum"),
          event_rate_Hz=("event_rate_Hz", "first"),
          mean_integral_mV_ns=("integral_mV_ns", "mean"),
          median_integral_mV_ns=("integral_mV_ns", "median"),
          mean_charge_pC=("charge_C", lambda x: np.mean(x) * 1e12),
          mean_charge_over_thr_pC=("charge_C", lambda x: np.mean(x[x > 0]) * 1e12 if np.any(x > 0) else np.nan),
          mean_I_A=("mean_anode_current_A", "mean"),
          mean_I_uA=("mean_anode_current_uA", "mean"),
          mean_I_over_thr_uA=("mean_anode_current_uA", lambda x: np.mean(x[x > 0]) if np.any(x > 0) else np.nan),
          mean_peak_mA=("peak_current_mA", "mean"),
          max_peak_mA=("peak_current_mA", "max"),
      )
      .reset_index()
      .sort_values(["hv_V", "channel"])
)

summary.to_csv(SUMMARY_CSV, index=False)


# ============================================================
# DIAGNOSI SATURAZIONE
# ============================================================

diagnosis_rows = []

for _, row in summary.iterrows():
    status_all, ratio_all, note_all = classify_saturation(row["mean_I_A"])
    mean_I_over_thr_A = row["mean_I_over_thr_uA"] * 1e-6 if pd.notnull(row["mean_I_over_thr_uA"]) else np.nan

    if pd.notnull(mean_I_over_thr_A):
        status_thr, ratio_thr, note_thr = classify_saturation(mean_I_over_thr_A)
    else:
        status_thr, ratio_thr, note_thr = "NO SIGNAL", np.nan, "no over-threshold events"

    diagnosis_rows.append({
        "run": row["run"],
        "hv_V": row["hv_V"],
        "channel": row["channel"],
        "n_events": row["n_events"],
        "n_over_threshold": row["n_over_threshold"],
        "mean_I_uA_all_events": row["mean_I_uA"],
        "mean_I_uA_over_threshold": row["mean_I_over_thr_uA"],
        "mean_peak_mA": row["mean_peak_mA"],
        "max_peak_mA": row["max_peak_mA"],
        "ratio_all_events_to_divider": ratio_all,
        "ratio_over_threshold_to_divider": ratio_thr,
        "status_all_events": status_all,
        "status_over_threshold": status_thr,
        "datasheet_dc_limit_uA": DC_LINEAR_OUTPUT_A * 1e6,
        "divider_current_mA": DIVIDER_CURRENT_A * 1e3,
        "handbook_1pct_limit_uA": DIVIDER_CURRENT_A * HANDBOOK_1PCT_RATIO * 1e6,
        "practical_20x_limit_uA": DIVIDER_CURRENT_A * PRACTICAL_20X_RATIO * 1e6,
        "practical_10x_limit_uA": DIVIDER_CURRENT_A * PRACTICAL_10X_RATIO * 1e6,
        "note_all_events": note_all,
        "note_over_threshold": note_thr,
    })

diagnosis = pd.DataFrame(diagnosis_rows)
diagnosis.to_csv(DIAGNOSIS_CSV, index=False)


# ============================================================
# OUTPUT TABELLE
# ============================================================

print("\n=== SUMMARY ===")
display(summary)

print("\n=== DIAGNOSIS ===")
display(diagnosis)


# ============================================================
# STAMPA CONCLUSIONE AUTOMATICA
# ============================================================

print("\n========== CONCLUSIONE AUTOMATICA ==========\n")

for _, row in diagnosis.sort_values(["hv_V", "channel"]).iterrows():
    print(
        f"HV = {int(row['hv_V'])} V | Ch {int(row['channel'])} | "
        f"I_mean(all) = {row['mean_I_uA_all_events']:.3f} uA | "
        f"I_mean(OT) = {row['mean_I_uA_over_threshold']:.3f} uA "
        if pd.notnull(row['mean_I_uA_over_threshold']) else
        f"HV = {int(row['hv_V'])} V | Ch {int(row['channel'])} | "
        f"I_mean(all) = {row['mean_I_uA_all_events']:.3f} uA | "
        f"I_mean(OT) = NaN "
    )

    print(
        f"   Divider current = {row['divider_current_mA']:.3f} mA | "
        f"Handbook 1% limit = {row['handbook_1pct_limit_uA']:.2f} uA | "
        f"Practical 20x limit = {row['practical_20x_limit_uA']:.2f} uA | "
        f"Practical 10x limit = {row['practical_10x_limit_uA']:.2f} uA | "
        f"Datasheet DC limit = {row['datasheet_dc_limit_uA']:.2f} uA"
    )

    print(
        f"   STATUS(all events): {row['status_all_events']} | {row['note_all_events']}"
    )
    print(
        f"   STATUS(over-threshold): {row['status_over_threshold']} | {row['note_over_threshold']}"
    )
    print(
        f"   Peak current: mean = {row['mean_peak_mA']:.5f} mA | max = {row['max_peak_mA']:.5f} mA\n"
    )


# ============================================================
# PLOT: CORRENTE MEDIA vs LIMITI
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

for ch in sorted(diagnosis["channel"].unique()):
    sel = diagnosis[diagnosis["channel"] == ch].sort_values("hv_V")

    ax.plot(
        sel["hv_V"],
        sel["mean_I_uA_all_events"],
        marker="o",
        linewidth=1.8,
        label=f"Ch {ch} | I_mean all events"
    )

    ax.plot(
        sel["hv_V"],
        sel["mean_I_uA_over_threshold"],
        marker="s",
        linestyle="--",
        linewidth=1.5,
        label=f"Ch {ch} | I_mean over-threshold"
    )

ax.axhline(DIVIDER_CURRENT_A * HANDBOOK_1PCT_RATIO * 1e6, color="black", linestyle="dotted", linewidth=1,
           label="Handbook 1% linearity limit")
ax.axhline(DIVIDER_CURRENT_A * PRACTICAL_20X_RATIO * 1e6, color="black", linestyle="--", linewidth=1,
           label="Practical 20x limit")
ax.axhline(DIVIDER_CURRENT_A * PRACTICAL_10X_RATIO * 1e6, color="black", linestyle="dashdot", linewidth=1,
           label="Practical 10x limit")
ax.axhline(DC_LINEAR_OUTPUT_A * 1e6, color="black", linestyle="solid", linewidth=1,
           label="Datasheet DC limit")

ax.set_xlabel("HV [V]")
ax.set_ylabel("Equivalent anode current [uA]")
ax.set_title("Equivalent anode current vs divider/datasheet limits")
ax.grid(True, alpha=0.3)
ax.set_ylim(0,200)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
